<a href="https://colab.research.google.com/github/MauricioLlugdar/CoffeeShop-Sales/blob/main/01_CoffeeShopSales_Analysis_Exploration_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coffee Shop Sales (Dirty Dataset) - Dataset Analysis and Exploration

## Plan de trabajo:

**Etapa inicial:** sería ver de analizar bien la data, hacer una limpieza y feature engineering.

**Accion**: Luego de la etapa inicial, el plan es predecir la cantidad vendida de la semana siguiente. Para esto deberemos crear las features necesarias para la predicción.

**Extra**: Lo ideal sería analizar cuanta cantidad se va a vender de cada articulo, para de esta manera saber cuanto stock se necesitará.

In [ ]:
!pip install kagglehub
import kagglehub
import pandas as pd

In [ ]:
path = kagglehub.dataset_download("ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cafe-sales-dirty-data-for-cleaning-training' dataset.
Path to dataset files: /kaggle/input/cafe-sales-dirty-data-for-cleaning-training


In [ ]:
df_raw = pd.read_csv(path + "/dirty_cafe_sales.csv")
df = df_raw.copy()
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Analisis general del dataset

In [ ]:
df.shape

(10000, 8)

In [ ]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

In [ ]:
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_9226047,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [ ]:
df.isna().mean().sort_values(ascending=False)

,0
Location,0.3265
Payment Method,0.2579
Item,0.0333
Price Per Unit,0.0179
Total Spent,0.0173
Transaction Date,0.0159
Quantity,0.0138
Transaction ID,0.0000


In [ ]:
print(df.duplicated().sum())


0


### Analisis de cada feature

In [ ]:
print(df.nunique().sort_values())

Location                4
Payment Method          5
Quantity                7
Price Per Unit          8
Item                   10
Total Spent            19
Transaction Date      367
Transaction ID      10000
dtype: int64


In [ ]:
exclude_cols = ['Transaction ID', 'Transaction Date']

for col in df.columns:
    if col not in exclude_cols:
        unique_vals = df[col].unique()
        print(f"{col} ({len(unique_vals)} valores únicos):\n {unique_vals}\n")
    else:
        unique_vals = df[col].unique()[:5]
        print(f"{col} (Mostrando 5 de {len(df[col].unique())}):\n {unique_vals}...\n")

Transaction ID (Mostrando 5 de 10000):
 ['TXN_1961373' 'TXN_4977031' 'TXN_4271903' 'TXN_7034554' 'TXN_3160411']...

Item (11 valores únicos):
 ['Coffee' 'Cake' 'Cookie' 'Salad' 'Smoothie' 'UNKNOWN' 'Sandwich' nan
 'ERROR' 'Juice' 'Tea']

Quantity (8 valores únicos):
 ['2' '4' '5' '3' '1' 'ERROR' 'UNKNOWN' nan]

Price Per Unit (9 valores únicos):
 ['2.0' '3.0' '1.0' '5.0' '4.0' '1.5' nan 'ERROR' 'UNKNOWN']

Total Spent (20 valores únicos):
 ['4.0' '12.0' 'ERROR' '10.0' '20.0' '9.0' '16.0' '15.0' '25.0' '8.0' '5.0'
 '3.0' '6.0' nan 'UNKNOWN' '2.0' '1.0' '7.5' '4.5' '1.5']

Payment Method (6 valores únicos):
 ['Credit Card' 'Cash' 'UNKNOWN' 'Digital Wallet' 'ERROR' nan]

Location (5 valores únicos):
 ['Takeaway' 'In-store' 'UNKNOWN' nan 'ERROR']

Transaction Date (Mostrando 5 de 368):
 ['2023-09-08' '2023-05-16' '2023-07-19' '2023-04-27' '2023-06-11']...



In [ ]:
df["Transaction ID"].nunique()


10000

In [ ]:
df["Transaction ID"].duplicated().sum()

np.int64(0)

Como no hay transacciones duplicadas, ni errores de carga, entonces elimino esta feature, ya que no va a ser de ayuda en la predicción

In [ ]:
df = df.drop(columns=["Transaction ID"])

Lo mejor que se puede hacer con las variables categoricas es aplicarles One Hot Encoding para que el modelo pueda predecir gracias a las mismas. Las variables categoricas que poseemos están dadas por las features:


*   Item
*   Payment Method
*   Location

